In [1]:
!pip install fastapi uvicorn pyngrok python-multipart nest_asyncio transformers torch pillow

In [ ]:
import pickle
from collections import deque
from typing import Dict, List, Optional

import numpy as np
import torch
import torch.nn as nn
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field


FRAUD_MODEL_PATH = "/content/statistical_fraud_model.pkl"
IOT_MODEL_PATH = "/content/iot_lstm_full_bundle.pt"

with open(FRAUD_MODEL_PATH, "rb") as f:
    fraud_bundle = pickle.load(f)

iot_bundle = torch.load(IOT_MODEL_PATH, map_location="cpu", weights_only=False)

# Fraud model config - updated with new structure
FRAUD_THRESHOLD = float(fraud_bundle.get("threshold", 14.27))
FRAUD_ZSCORE_FEATURES = list(fraud_bundle.get("zscore_features", ["amt", "distance"]))
FRAUD_EXTRA_FEATURES = list(fraud_bundle.get("extra_features", ["city_pop", "ema_diff"]))
FRAUD_ROLLING_WINDOW = int(fraud_bundle.get("rolling_window", 50))
FRAUD_EMA_SPAN = int(fraud_bundle.get("ema_span", 20))
FRAUD_ALPHA = 2.0 / (FRAUD_EMA_SPAN + 1.0)

    # Weights for combined scoring (from best_weights tuple)
    # best_weights = (w_amt, w_dist, w_city, w_ema)
    best_weights = fraud_bundle.get("best_weights", (0.5, 0.5, 0.5, 3.0))
    FRAUD_AMT_WEIGHT = float(best_weights[0])
    FRAUD_DIST_WEIGHT = float(best_weights[1])
    FRAUD_CITY_WEIGHT = float(best_weights[2])
    FRAUD_EMA_DIFF_WEIGHT = float(best_weights[3])
    
    # DEBUG: Show loaded weights
    print(f"Weights loaded: amt={FRAUD_AMT_WEIGHT}, dist={FRAUD_DIST_WEIGHT}, city={FRAUD_CITY_WEIGHT}, ema_diff={FRAUD_EMA_DIFF_WEIGHT}")

    # Statistical params for city_pop and ema_diff
    FRAUD_CITY_MU = float(fraud_bundle.get("city_mu", 89119.20))
    FRAUD_CITY_SD = float(fraud_bundle.get("city_sd", 302553.86))
    FRAUD_EMA_MU = float(fraud_bundle.get("ema_mu", 60.92))
    FRAUD_EMA_SD = float(fraud_bundle.get("ema_sd", 153.92))

    # Scaler params for city_pop (STATIC distribution - global mean/std from training)
    # These are used for city_pop z-score calculation
    FRAUD_CITY_MU = float(fraud_bundle.get("city_mu", 89119.20))  # From training data
    FRAUD_CITY_SD = float(fraud_bundle.get("city_sd", 302553.86))  # From training data
    FRAUD_EMA_MU = float(fraud_bundle.get("ema_mu", 60.92))  # From training data
    FRAUD_EMA_SD = float(fraud_bundle.get("ema_sd", 153.92))  # From training data
    
    print(f"Loaded scaler params - city: ({FRAUD_CITY_MU:.2f}, {FRAUD_CITY_SD:.2f})")
    print(f"Loaded scaler params - ema: ({FRAUD_EMA_MU:.2f}, {FRAUD_EMA_SD:.2f})")


IOT_THRESHOLD = float(iot_bundle.get("threshold", 0.45))
IOT_WINDOW_SIZE = int(iot_bundle.get("window_size", 80))
IOT_FEATURES = list(iot_bundle.get("features", []))
IOT_SCALER = iot_bundle.get("scaler")
IOT_STATE_DICT = iot_bundle.get("model_state_dict", {})


class SequenceLSTMClassifier(nn.Module):
    def __init__(self, input_size: int, hidden_size: int = 256, num_layers: int = 3):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True
        )
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, sequence: torch.Tensor) -> torch.Tensor:
        output, _ = self.lstm(sequence)
        return self.fc(output[:, -1, :])


hidden_size = IOT_STATE_DICT["fc.weight"].shape[1]
num_layers = len([k for k in IOT_STATE_DICT.keys() if k.startswith("lstm.weight_ih_")])

iot_model = SequenceLSTMClassifier(
    input_size=len(IOT_FEATURES),
    hidden_size=hidden_size,
    num_layers=num_layers
)
iot_model.load_state_dict(IOT_STATE_DICT)
iot_model.eval()


SESSION_STATE: Dict[str, dict] = {}


def get_session_state(session_id: str) -> dict:
    if session_id not in SESSION_STATE:
        SESSION_STATE[session_id] = {
            "fraud": {
                "stat_windows": {},
                "ema_values": {},
            },
            "iot": {
                "sequence_buffer": deque(maxlen=IOT_WINDOW_SIZE),
            }
        }
    return SESSION_STATE[session_id]


class FraudPredictRequest(BaseModel):
    session_id: str
    amt: float
    distance: float
    city_pop: float
    threshold_override: Optional[float] = None


class IoTPredictRequest(BaseModel):
    session_id: str
    Temperature: float
    Humidity: float
    Battery_Level: float
    temp_diff: float
    hum_diff: float
    bat_diff: float
    temp_mean10: float
    temp_std10: float
    hum_mean10: float
    hum_std10: float
    threshold_override: Optional[float] = None


class PredictResponse(BaseModel):
    session_id: str
    stream_type: str
    model_key: str
    is_anomaly: bool
    anomaly_score: float
    threshold: float
    reasons: List[str] = Field(default_factory=list)
    metadata: Dict = Field(default_factory=dict)


app = FastAPI(title="Project2 Colab AI Inference Service")


@app.get("/")
def root():
    return {
        "service": "project2-colab-inference",
        "routes": ["/health", "/models", "/predict/fraud", "/predict/iot", "/sessions/reset/{session_id}"]
    }


@app.get("/health")
def health():
    return {
        "status": "ok",
        "loaded_models": ["fraud_statistical", "iot_lstm"],
        "fraud_config": {
            "threshold": FRAUD_THRESHOLD,
            "zscore_features": FRAUD_ZSCORE_FEATURES,
            "extra_features": FRAUD_EXTRA_FEATURES,
            "rolling_window": FRAUD_ROLLING_WINDOW,
            "ema_span": FRAUD_EMA_SPAN,
            "weights": {
                "amt": FRAUD_AMT_WEIGHT,
                "distance": FRAUD_DIST_WEIGHT,
                "city": FRAUD_CITY_WEIGHT,
                "ema_diff": FRAUD_EMA_DIFF_WEIGHT
            }
        },
        "iot_config": {
            "threshold": IOT_THRESHOLD,
            "window_size": IOT_WINDOW_SIZE,
            "features": IOT_FEATURES,
        },
        "active_sessions": len(SESSION_STATE),
    }


@app.get("/models")
def models():
    return [
        {
            "model_key": "fraud_statistical",
            "stream_type": "credit_card",
            "threshold": FRAUD_THRESHOLD,
            "best_f1": float(fraud_bundle.get("best_f1", 0.378)),
            "zscore_features": FRAUD_ZSCORE_FEATURES,
            "extra_features": FRAUD_EXTRA_FEATURES,
            "rolling_window": FRAUD_ROLLING_WINDOW,
            "ema_span": FRAUD_EMA_SPAN,
            "weights": {
                "amt": FRAUD_AMT_WEIGHT,
                "distance": FRAUD_DIST_WEIGHT,
                "city": FRAUD_CITY_WEIGHT,
                "ema_diff": FRAUD_EMA_DIFF_WEIGHT
            },
            "statistical_params": {
                "city_pop": {"mu": FRAUD_CITY_MU, "sd": FRAUD_CITY_SD},
                "ema_diff": {"mu": FRAUD_EMA_MU, "sd": FRAUD_EMA_SD}
            }
        },
        {
            "model_key": "iot_lstm",
            "stream_type": "iot_sensor",
            "threshold": IOT_THRESHOLD,
            "window_size": IOT_WINDOW_SIZE,
            "features": IOT_FEATURES,
        }
    ]


@app.post("/sessions/reset/{session_id}")
def reset_session(session_id: str):
    if session_id in SESSION_STATE:
        del SESSION_STATE[session_id]
    return {"message": "session reset", "session_id": session_id}


def update_fraud_feature(runtime_state: dict, feature_name: str, value: float):
    window_map = runtime_state.setdefault("stat_windows", {})
    ema_map = runtime_state.setdefault("ema_values", {})

    window = window_map.setdefault(feature_name, deque(maxlen=FRAUD_ROLLING_WINDOW))

    if len(window) > 0:
        mean_value = sum(window) / len(window)
        variance = sum((item - mean_value) ** 2 for item in window) / len(window)
        std_value = variance ** 0.5
    else:
        mean_value = value
        std_value = 0.0

    prev_ema = ema_map.get(feature_name, value)
    # Calculate new EMA (for NEXT iteration, NOT current - matches .shift(1) in notebook)
    new_ema = FRAUD_ALPHA * value + (1.0 - FRAUD_ALPHA) * prev_ema
    # Store new EMA for next call
    ema_map[feature_name] = new_ema

    window.append(value)

    zscore = 0.0 if std_value == 0 else abs((value - mean_value) / std_value)
    # Return PREVIOUS EMA (shifted - matches notebook's .shift(1))
    return zscore, prev_ema


@app.post("/predict/fraud", response_model=PredictResponse)
def predict_fraud(payload: FraudPredictRequest):
    session_state = get_session_state(payload.session_id)["fraud"]

    # Get raw input values
    raw_amt = float(payload.amt)
    raw_distance = float(payload.distance)
    raw_city_pop = float(payload.city_pop)

    reasons = []
    amt_zscore_contrib = 0.0
    dist_zscore_contrib = 0.0
    ema_gap = 0.0

    # Process amt using ROLLING z-score with its own weight
    amt_zscore, amt_ema = update_fraud_feature(session_state, "amt", raw_amt)
    amt_zscore_contrib = amt_zscore * FRAUD_AMT_WEIGHT
    ema_gap += abs(raw_amt - amt_ema)
    if amt_zscore >= 3.0:
        reasons.append("amt_zscore_high")

    # Process distance using ROLLING z-score with its own weight
    dist_zscore, dist_ema = update_fraud_feature(session_state, "distance", raw_distance)
    dist_zscore_contrib = dist_zscore * FRAUD_DIST_WEIGHT

    # Process city_pop using STATIC z-score (global mean/std from training)
    city_zscore = abs(raw_city_pop - FRAUD_CITY_MU) / (FRAUD_CITY_SD + 1e-10)
    city_contrib = city_zscore * FRAUD_CITY_WEIGHT

    # Calculate ema_diff contribution (NORMALIZED like in training)
    # In training: ema_diff_z = |ema_diff - ema_mu| / ema_sd
    ema_diff_z = abs(ema_gap - FRAUD_EMA_MU) / (FRAUD_EMA_SD + 1e-10)
    ema_diff_contrib = ema_diff_z * FRAUD_EMA_DIFF_WEIGHT

    # Combined score (matches training: w_amt*amt_z + w_dist*dist_z + w_city*city_z + w_ema*ema_diff_z)
    score = float(amt_zscore_contrib + dist_zscore_contrib + city_contrib + ema_diff_contrib)
    
    # DEBUG: Log detailed breakdown
    print(f"[DEBUG] amt_z={amt_zscore:.4f}, dist_z={dist_zscore:.4f}, city_z={city_zscore:.4f}, ema_diff_z={ema_diff_z:.4f}")
    print(f"[DEBUG] amt_c={amt_zscore_contrib:.4f}, dist_c={dist_zscore_contrib:.4f}, city_c={city_contrib:.4f}, ema_c={ema_diff_contrib:.4f}")
    print(f"[DEBUG] score={score:.4f}, threshold={FRAUD_THRESHOLD:.4f}")

    # Threshold from model (original: ~14.27)
    threshold = float(payload.threshold_override if payload.threshold_override is not None else FRAUD_THRESHOLD)
    is_anomaly = score >= threshold

    if score >= threshold * 1.2:
        reasons.append("score_far_above_threshold")
    if not reasons and is_anomaly:
        reasons.append("statistical_threshold_exceeded")

    return PredictResponse(
        session_id=payload.session_id,
        stream_type="credit_card",
        model_key="fraud_statistical",
        is_anomaly=bool(is_anomaly),
        anomaly_score=score,
        threshold=threshold,
        reasons=reasons,
        metadata={
            "raw_amt": raw_amt,
            "raw_distance": raw_distance,
            "raw_city_pop": raw_city_pop,
            "amt_zscore": amt_zscore,
            "dist_zscore": dist_zscore,
            "city_zscore": city_zscore,
            "ema_diff_z": ema_diff_z,
            "score_contrib": {
                "amt": amt_zscore_contrib,
                "distance": dist_zscore_contrib,
                "city": city_contrib,
                "ema_diff": ema_diff_contrib
            },
            "weights": {
                "amt": FRAUD_AMT_WEIGHT,
                "distance": FRAUD_DIST_WEIGHT,
                "city": FRAUD_CITY_WEIGHT,
                "ema_diff": FRAUD_EMA_DIFF_WEIGHT
            }
        }
    )

@app.post("/predict/iot", response_model=PredictResponse)
def predict_iot(payload: IoTPredictRequest):
    session_state = get_session_state(payload.session_id)["iot"]
    sequence_buffer = session_state.setdefault("sequence_buffer", deque(maxlen=IOT_WINDOW_SIZE))

    feature_vector = [
        float(payload.Temperature),
        float(payload.Humidity),
        float(payload.Battery_Level),
        float(payload.temp_diff),
        float(payload.hum_diff),
        float(payload.bat_diff),
        float(payload.temp_mean10),
        float(payload.temp_std10),
        float(payload.hum_mean10),
        float(payload.hum_std10),
    ]

    sequence_buffer.append(feature_vector)

    if len(sequence_buffer) < IOT_WINDOW_SIZE:
        return PredictResponse(
            session_id=payload.session_id,
            stream_type="iot_sensor",
            model_key="iot_lstm",
            is_anomaly=False,
            anomaly_score=0.0,
            threshold=float(payload.threshold_override if payload.threshold_override is not None else IOT_THRESHOLD),
            reasons=["warming_up_window"],
            metadata={
                "window_progress": len(sequence_buffer),
                "window_size": IOT_WINDOW_SIZE,
            }
        )

    sequence = np.array(sequence_buffer, dtype=np.float32)
    scaled = IOT_SCALER.transform(sequence)
    tensor = torch.tensor(scaled, dtype=torch.float32).unsqueeze(0)

    with torch.no_grad():
        logit = iot_model(tensor)
        probability = float(torch.sigmoid(logit).item())

    threshold = float(payload.threshold_override if payload.threshold_override is not None else IOT_THRESHOLD)
    is_anomaly = probability >= threshold
    reasons = ["iot_sequence_probability_high"] if is_anomaly else []

    return PredictResponse(
        session_id=payload.session_id,
        stream_type="iot_sensor",
        model_key="iot_lstm",
        is_anomaly=bool(is_anomaly),
        anomaly_score=probability,
        threshold=threshold,
        reasons=reasons,
        metadata={"window_size": IOT_WINDOW_SIZE}
    )

In [3]:
from pyngrok import ngrok
import nest_asyncio

NGROK_AUTH_TOKEN = "3BZChrken9p8PVqmTWWvHdifWru_77shUGwQTNwYsEUpLxZw5"
ngrok.set_auth_token(NGROK_AUTH_TOKEN)

nest_asyncio.apply()
public_url = ngrok.connect(8000)
print("Public URL:", public_url)

Public URL: NgrokTunnel: "https://subtriquetrous-jaggedly-kelsey.ngrok-free.dev" -> "http://localhost:8000"


In [ ]:
import asyncio
import uvicorn

config = uvicorn.Config(app, host="0.0.0.0", port=8000)
server = uvicorn.Server(config)

await server.serve()

INFO:     Started server process [36867]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)
